# 10 â€” Leaderboard
Assembles baselines + default-hyperparam CV + tuned CV. Winner per target by primary metric (MAE min / PR-AUC max).

In [1]:

import sys, os
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
import json
import numpy as np
import pandas as pd
from src.config import RESULTS_DIR, TARGET_CLF

REG_COLS = ["model", "target", "mae_mean", "rmse_mean", "r2_mean"]
CLF_COLS = ["model", "target", "pr_auc_mean", "pr_auc_std", "f1_mean", "roc_auc_mean"]

base_reg = pd.read_csv(RESULTS_DIR / "baselines_reg_cv.csv")
rows = []
for _, r in base_reg.iterrows():
    rows.append({"model": "baseline_mean", "target": r["target"],
                 "mae_mean": r["mae_mean_mean"], "rmse_mean": r["rmse_mean_mean"], "r2_mean": r["r2_mean_mean"]})
    rows.append({"model": "baseline_persistence", "target": r["target"],
                 "mae_mean": r["mae_mean_persistence"], "rmse_mean": r["rmse_mean_persistence"], "r2_mean": r["r2_mean_persistence"]})
frames = [pd.DataFrame(rows)]
frames.append(pd.read_csv(RESULTS_DIR / "reg_baseline_cv.csv")[REG_COLS])
tuned_paths = sorted(RESULTS_DIR.glob("reg_tuned_cv_*.csv"))
if tuned_paths:
    frames.append(pd.concat([pd.read_csv(p)[REG_COLS] for p in tuned_paths], ignore_index=True))
leaderboard_reg = pd.concat(frames, ignore_index=True)
leaderboard_reg.to_csv(RESULTS_DIR / "leaderboard_regression.csv", index=False)
display(leaderboard_reg.sort_values(["target", "mae_mean"]))


,model,target,mae_mean,rmse_mean,r2_mean
17,random_forest_tuned,next_cycle_length,1.19160,1.73416,0.79573
18,xgboost_tuned,next_cycle_length,1.20509,1.75389,0.79109
19,lightgbm_tuned,next_cycle_length,1.21017,1.76233,0.78908
5,random_forest,next_cycle_length,1.21759,1.77032,0.78722
16,ridge_tuned,next_cycle_length,1.22316,1.78756,0.78301
4,ridge,next_cycle_length,1.22345,1.78799,0.78290
20,svr_tuned,next_cycle_length,1.23922,1.78624,0.78329
8,svr,next_cycle_length,1.24341,1.79129,0.78208
7,lightgbm,next_cycle_length,1.26114,1.83171,0.77207
21,knn_tuned,next_cycle_length,1.27760,1.82624,0.77347


In [3]:
winners_reg = (leaderboard_reg.sort_values("mae_mean").groupby("target").first()[["model", "mae_mean"]])
print("REGRESSION WINNERS")
display(winners_reg)


REGRESSION WINNERS


,model,mae_mean
target,,
next_cycle_length,random_forest_tuned,1.19160
next_period_length,random_forest_tuned,0.57132


## Classification leaderboard

In [4]:
base_clf = pd.read_csv(RESULTS_DIR / "baselines_clf_cv.csv")
base_clf = base_clf.rename(columns={c: f"{c}_mean" for c in ["pr_auc", "f1", "roc_auc"]})
base_clf["target"] = TARGET_CLF
frames_c = [base_clf.reindex(columns=CLF_COLS)]
frames_c.append(pd.read_csv(RESULTS_DIR / "clf_baseline_cv.csv")[CLF_COLS])
tuned_path = RESULTS_DIR / f"clf_tuned_cv_{TARGET_CLF}.csv"
if tuned_path.exists():
    frames_c.append(pd.read_csv(tuned_path)[CLF_COLS])
leaderboard_clf = pd.concat(frames_c, ignore_index=True)
leaderboard_clf.to_csv(RESULTS_DIR / "leaderboard_classification.csv", index=False)
display(leaderboard_clf.sort_values("pr_auc_mean", ascending=False))

winner_clf = leaderboard_clf.sort_values("pr_auc_mean", ascending=False).iloc[0]
print("CLASSIFICATION WINNER:", winner_clf["model"], "pr_auc:", winner_clf["pr_auc_mean"])
with open(RESULTS_DIR / "winners.json", "w") as f:
    json.dump({
        "regression": winners_reg.reset_index().to_dict(orient="records"),
        "classification": {k: (float(v) if isinstance(v, (np.floating,)) else v) for k, v in winner_clf.items()},
    }, f, indent=2)


,model,target,pr_auc_mean,pr_auc_std,f1_mean,roc_auc_mean
9,random_forest_tuned,next_is_irregular,0.87767,0.02137,0.75029,0.97060
3,random_forest,next_is_irregular,0.87266,0.02337,0.79219,0.96897
13,knn_tuned,next_is_irregular,0.86822,0.02924,0.78058,0.96122
11,lightgbm_tuned,next_is_irregular,0.86652,0.02383,0.78725,0.96330
10,xgboost_tuned,next_is_irregular,0.86018,0.02993,0.71938,0.96489
5,lightgbm,next_is_irregular,0.85117,0.02303,0.78153,0.95723
4,xgboost,next_is_irregular,0.84228,0.02390,0.77739,0.95292
12,svc_tuned,next_is_irregular,0.83361,0.01966,0.78167,0.95928
6,svc,next_is_irregular,0.81211,0.04107,0.77999,0.95790
7,knn,next_is_irregular,0.78186,0.04307,0.78033,0.91431


CLASSIFICATION WINNER: random_forest_tuned pr_auc: 0.87767
